# Cross-Parameter Comparison: Cell-Calling Threshold Sweep

Compare how different cell-calling UMI thresholds affect QC metrics, knockdown efficiency,
and DEG detection across the 5 benchmark technologies.

**Runs compared:**
- `cleanser_500_mito_15pc` — 500 UMI threshold (3 datasets)
- `cleanser_800_mito_15pc` — 800 UMI threshold (5 datasets)
- `cleanser_knee2_mito_15pc` — knee inflection threshold (4 datasets)
- `cleanser_unified` — original unified run (5 datasets, reference)

**Key questions:**
1. Cell yield vs quality tradeoff
2. Knockdown sensitivity across thresholds
3. DEG robustness
4. Optimal threshold per technology

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
import sys
import yaml
import warnings
warnings.filterwarnings('ignore')

In [ ]:
PROJECT_ROOT = Path("/cellar/users/aklie/projects/tf_perturb_seq")
BASE_DIR = PROJECT_ROOT / "datasets" / "technology-benchmark_WTC11_TF-Perturb-seq"
RESULTS_DIR = BASE_DIR / "results" / "cross_tech_comparison"
OUTPUT_DIR = RESULTS_DIR / "cross_parameter"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(PROJECT_ROOT / "config"))
from loader import load_colors
colors_cfg = load_colors("technology-benchmark_WTC11_TF-Perturb-seq")
dataset_colors = colors_cfg['dataset_colors']
dataset_order = colors_cfg['dataset_order']

# Runs to compare (label → display name)
RUNS = {
    'cleanser_500_mito_15pc':  {'label': '500 UMI',  'color': '#1b9e77', 'marker': 'o'},
    'cleanser_800_mito_15pc':  {'label': '800 UMI',  'color': '#d95f02', 'marker': 's'},
    'cleanser_knee2_mito_15pc':{'label': 'Knee',     'color': '#7570b3', 'marker': '^'},
    'cleanser_unified':        {'label': 'Unified',  'color': '#e7298a', 'marker': 'D'},
}
run_labels = list(RUNS.keys())

# Short dataset names for plotting
SHORT_NAMES = {
    'Hon_WTC11-benchmark_TF-Perturb-seq': 'Hon',
    'Huangfu_WTC11-benchmark_TF-Perturb-seq': 'Huangfu',
    'Gersbach_WTC11-benchmark_TF-Perturb-seq_GEM-Xv3': 'Gersbach\nGEM-Xv3',
    'Gersbach_WTC11-benchmark_TF-Perturb-seq_HTv2': 'Gersbach\nHTv2',
    'Engreitz_WTC11-benchmark_TF-Perturb-seq': 'Engreitz',
}
print(f"Output: {OUTPUT_DIR}")

## 1. Load data from all runs

In [ ]:
# Load metrics_summary.tsv from each run
summary_dfs = []
target_dfs = []
target_metrics_dfs = []

for run in run_labels:
    run_dir = RESULTS_DIR / run
    
    # Metrics summary
    f = run_dir / 'metrics_summary.tsv'
    if f.exists():
        df = pd.read_csv(f, sep='\t')
        df['run'] = run
        df['run_label'] = RUNS[run]['label']
        summary_dfs.append(df)
        print(f"  Metrics [{run}]: {len(df)} datasets")
    
    # Intended target results
    f = run_dir / 'combined_intended_target_results.tsv'
    if f.exists():
        df = pd.read_csv(f, sep='\t')
        df['run'] = run
        df['run_label'] = RUNS[run]['label']
        target_dfs.append(df)
        print(f"  Target results [{run}]: {len(df)} guides")
    
    # Intended target metrics (auROC, auPRC)
    f = run_dir / 'combined_intended_target_metrics.tsv'
    if f.exists():
        df = pd.read_csv(f, sep='\t')
        df['run'] = run
        df['run_label'] = RUNS[run]['label']
        target_metrics_dfs.append(df)
        print(f"  Target metrics [{run}]: {len(df)} datasets")

summary_df = pd.concat(summary_dfs, ignore_index=True)
summary_df['short_name'] = summary_df['dataset'].map(SHORT_NAMES)

if target_dfs:
    target_df = pd.concat(target_dfs, ignore_index=True)
    target_df['short_name'] = target_df['dataset'].map(SHORT_NAMES)

if target_metrics_dfs:
    target_metrics_df = pd.concat(target_metrics_dfs, ignore_index=True)
    target_metrics_df['short_name'] = target_metrics_df['dataset'].map(SHORT_NAMES)

print(f"\nTotal: {len(summary_df)} dataset-run combinations")

## 2. QC Metrics Across Thresholds

In [ ]:
# Plot key QC metrics as grouped bar charts: x=dataset, groups=threshold
metrics_to_plot = [
    ('n_cells', 'Total Cells'),
    ('umi_median', 'Median UMI/Cell'),
    ('genes_median', 'Median Genes/Cell'),
    ('mito_median', 'Median % Mito'),
    ('guide_umi_median', 'Median Guide UMI/Cell'),
    ('frac_cells_with_guide', 'Fraction Cells\nwith Guide'),
]
metrics_to_plot = [(c, t) for c, t in metrics_to_plot if c in summary_df.columns]

n_plots = len(metrics_to_plot)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

ds_in_data = [ds for ds in dataset_order if ds in summary_df['dataset'].values]
n_ds = len(ds_in_data)
n_runs = len(run_labels)
width = 0.8 / n_runs

for ax_idx, (col, title) in enumerate(metrics_to_plot):
    ax = axes[ax_idx]
    x = np.arange(n_ds)
    for j, run in enumerate(run_labels):
        run_data = summary_df[summary_df['run'] == run]
        vals = []
        for ds in ds_in_data:
            row = run_data[run_data['dataset'] == ds]
            vals.append(row[col].values[0] if len(row) > 0 else np.nan)
        offset = (j - (n_runs - 1) / 2) * width
        bars = ax.bar(x + offset, vals, width, label=RUNS[run]['label'],
                      color=RUNS[run]['color'], alpha=0.85, edgecolor='white', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels([SHORT_NAMES.get(ds, ds) for ds in ds_in_data], fontsize=9)
    ax.set_title(title, fontsize=11)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if col == 'n_cells':
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

for ax in axes[n_plots:]:
    ax.set_visible(False)

axes[0].legend(loc='upper right', fontsize=9)
fig.suptitle('QC Metrics Across Cell-Calling Thresholds', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'qc_metrics_across_thresholds.pdf', dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'qc_metrics_across_thresholds.pdf'}")

## 3. Knockdown Efficiency Across Thresholds

In [ ]:
# Compare auROC and auPRC across thresholds
if target_metrics_dfs:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for ax, metric, title in zip(axes, ['auroc', 'auprc'], ['auROC', 'auPRC']):
        if metric not in target_metrics_df.columns:
            ax.set_visible(False)
            continue
        ds_in = [ds for ds in dataset_order if ds in target_metrics_df['dataset'].values]
        x = np.arange(len(ds_in))
        for j, run in enumerate(run_labels):
            run_data = target_metrics_df[target_metrics_df['run'] == run]
            vals = []
            for ds in ds_in:
                row = run_data[run_data['dataset'] == ds]
                vals.append(row[metric].values[0] if len(row) > 0 else np.nan)
            offset = (j - (n_runs - 1) / 2) * width
            ax.bar(x + offset, vals, width, label=RUNS[run]['label'],
                   color=RUNS[run]['color'], alpha=0.85, edgecolor='white', linewidth=0.5)
        ax.set_xticks(x)
        ax.set_xticklabels([SHORT_NAMES.get(ds, ds) for ds in ds_in], fontsize=9)
        ax.set_title(title, fontsize=12)
        ax.set_ylim(0, 1.05)
        ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
    axes[0].legend(loc='lower right', fontsize=9)
    fig.suptitle('Knockdown Efficiency Across Thresholds', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'auroc_auprc_across_thresholds.pdf', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {OUTPUT_DIR / 'auroc_auprc_across_thresholds.pdf'}")

In [ ]:
# Compare log2FC distributions across thresholds for targeting guides
if target_dfs:
    # Filter to targeting guides only
    pos_data = target_df[target_df['targeting'] == True].copy()
    
    if 'log2_fc' in pos_data.columns and len(pos_data) > 0:
        ds_in = [ds for ds in dataset_order if ds in pos_data['dataset'].values]
        n_ds_in = len(ds_in)
        
        fig, axes = plt.subplots(1, n_ds_in, figsize=(4 * n_ds_in, 5), sharey=True)
        if n_ds_in == 1:
            axes = [axes]
        
        for ax, ds in zip(axes, ds_in):
            ds_data = pos_data[pos_data['dataset'] == ds]
            box_data = []
            box_labels = []
            box_colors = []
            for run in run_labels:
                run_data = ds_data[ds_data['run'] == run]['log2_fc'].dropna()
                if len(run_data) > 0:
                    box_data.append(run_data.values)
                    box_labels.append(RUNS[run]['label'])
                    box_colors.append(RUNS[run]['color'])
            if box_data:
                bp = ax.boxplot(box_data, labels=box_labels, patch_artist=True, widths=0.6)
                for patch, color in zip(bp['boxes'], box_colors):
                    patch.set_facecolor(color)
                    patch.set_alpha(0.7)
            ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
            ax.set_title(SHORT_NAMES.get(ds, ds), fontsize=11)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
        
        axes[0].set_ylabel('log2(FC)')
        fig.suptitle('Targeting Guide log2FC by Threshold', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'log2fc_across_thresholds.pdf', dpi=300, bbox_inches='tight')
        plt.show()
        print(f"Saved: {OUTPUT_DIR / 'log2fc_across_thresholds.pdf'}")

## 4. Per-Guide log2FC Correlation Across Thresholds

In [ ]:
# For each dataset, scatter per-guide log2FC across pairs of thresholds
from scipy.stats import spearmanr

if target_dfs:
    ds_in = [ds for ds in dataset_order if ds in target_df['dataset'].values]
    runs_in = [r for r in run_labels if r in target_df['run'].values]
    n_pairs = len(runs_in) * (len(runs_in) - 1) // 2
    
    for ds in ds_in:
        ds_data = target_df[target_df['dataset'] == ds]
        if 'log2_fc' not in ds_data.columns:
            continue
        
        # Pivot to get guide x run matrix
        guide_col = 'guide_id' if 'guide_id' in ds_data.columns else 'intended_target_name'
        pivot = ds_data.pivot_table(index=guide_col, columns='run', values='log2_fc', aggfunc='mean')
        
        pairs = [(i, j) for i in range(len(runs_in)) for j in range(i+1, len(runs_in))]
        if len(pairs) == 0:
            continue
        
        ncols = min(3, len(pairs))
        nrows = int(np.ceil(len(pairs) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows))
        if len(pairs) == 1:
            axes = np.array([axes])
        axes = np.atleast_2d(axes).flatten()
        
        for idx, (i, j) in enumerate(pairs):
            ax = axes[idx]
            r1, r2 = runs_in[i], runs_in[j]
            if r1 in pivot.columns and r2 in pivot.columns:
                shared = pivot[[r1, r2]].dropna()
                if len(shared) > 5:
                    rho, pval = spearmanr(shared[r1], shared[r2])
                    ax.scatter(shared[r1], shared[r2], s=10, alpha=0.5,
                              color=dataset_colors.get(ds, 'gray'))
                    lims = [min(shared[r1].min(), shared[r2].min()),
                           max(shared[r1].max(), shared[r2].max())]
                    ax.plot(lims, lims, 'k--', alpha=0.3)
                    ax.set_xlabel(RUNS[r1]['label'])
                    ax.set_ylabel(RUNS[r2]['label'])
                    ax.set_title(f'rho={rho:.3f} (n={len(shared)})', fontsize=10)
                    ax.spines['top'].set_visible(False)
                    ax.spines['right'].set_visible(False)
            else:
                ax.set_visible(False)
        
        for ax in axes[len(pairs):]:
            ax.set_visible(False)
        
        short = SHORT_NAMES.get(ds, ds)
        fig.suptitle(f'{short}: Per-Guide log2FC Across Thresholds', fontsize=13, fontweight='bold')
        plt.tight_layout()
        safe_name = ds.replace(' ', '_')
        plt.savefig(OUTPUT_DIR / f'log2fc_correlation_{safe_name}.pdf', dpi=300, bbox_inches='tight')
        plt.show()
        print(f"Saved: {short}")

## 5. Summary Table

In [ ]:
# Create a pivot summary: datasets x runs for key metrics
key_cols = ['n_cells', 'umi_median', 'guide_umi_median', 'frac_cells_with_guide']
key_cols = [c for c in key_cols if c in summary_df.columns]

for col in key_cols:
    pivot = summary_df.pivot_table(index='dataset', columns='run_label', values=col)
    pivot.index = pivot.index.map(lambda x: SHORT_NAMES.get(x, x))
    print(f"\n=== {col} ===")
    display(pivot.round(2))

# Save combined summary
summary_df.to_csv(OUTPUT_DIR / 'cross_parameter_summary.tsv', sep='\t', index=False)
print(f"\nSaved: {OUTPUT_DIR / 'cross_parameter_summary.tsv'}")